In [32]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [60]:
import matplotlib.pyplot as plt
import multiprocessing as mp

from collections import Counter
from gensim.models.word2vec import LineSentence
from hvectorspaces.io import PostgresClient
from hvectorspaces.analysis.preprocess import (
    merge_title_and_abstract,
    clean_openalex_text,
    top_hyphenated_words,
    print_counts,
    get_lemmas_and_non_stop_from_nlp,
    replace_hyphen_terms,
    init_spacy,
    spacy_docs,
    create_per_decade_index,
    split_corpus_by_decade
)

from hvectorspaces.analysis.gensim_models import (
    ngram_builder,
    write_corpus,
    EpochLogger,
    fit_model,
    save_model
)

from hvectorspaces.analysis.diagnostics import (
    sanity_check
)
from itertools import islice
from tqdm.notebook import tqdm

In [48]:
fields = [
    "title", 
    "publication_year", 
    "domain", 
    "field", 
    "topic",
    "abstract"
]

data_per_decade = list()
decades = list(range(1920,2030,10))

with PostgresClient() as client:
    for d in decades:
        citation_map = client.fetch_per_decade_data(d, additional_fields=fields)
        id_to_cited_ids = {}
        id_to_attrs = {} 
        for oa_id, refs, title, publication_year, domain, field, topic, abstract in citation_map:
            id_to_cited_ids[oa_id] = refs
            id_to_attrs[oa_id] = {
                "title": title,
                "publication_year": publication_year,
                "domain": domain,
                "field": field,
                "topic": topic,
                "abstract" : abstract
            }
        data_per_decade.append(id_to_attrs)

In [49]:
for d in data_per_decade:
    print(len(d))

10
32
63
410
1534
3462
6601
20160
82622
161255
60036


In [50]:
for k, v in islice(data_per_decade[-1].items(), 2):
    print(k, v)

W1019189097 {'title': 'canonical correlation forests', 'publication_year': 2022, 'domain': 'Physical Sciences', 'field': 'Computer Science', 'topic': 'Neural Networks and Applications', 'abstract': 'We introduce canonical correlation forests (CCFs), a new decision tree ensemble method for classification and regression. Individual canonical correlation trees are binary decision trees with hyperplane splits based on local canonical correlation coefficients calculated during training. Unlike axis-aligned alternatives, the decision surfaces of CCFs are not restricted to the coordinate system of the inputs features and therefore more naturally represent data with correlated inputs. CCFs naturally accommodate multiple outputs, provide a similar computational complexity to random forests, and inherit their impressive robustness to the choice of input parameters. As part of the CCF training algorithm, we also introduce projection bootstrapping, a novel alternative to bagging for oblique decisi

In [51]:
corpus_per_decade = dict()

for idx, d in enumerate(data_per_decade):
    txts = [merge_title_and_abstract(v) for v in d.values()]
    #print(txts[1])
    txts = [clean_openalex_text(entry) for entry in txts if isinstance(entry, str)]
    #print(txts[1])
    corpus_per_decade[decades[idx]] = txts
    print(decades[idx])
    print(str(len(txts)) + "\n")

1920
10

1930
32

1940
63

1950
410

1960
1534

1970
3462

1980
6601

1990
20156

2000
82605

2010
161251

2020
60036



In [52]:
corpus_per_decade[2020][0:2]

['canonical correlation forests. We introduce canonical correlation forests (CCFs), a new decision tree ensemble method for classification and regression. Individual canonical correlation trees are binary decision trees with hyperplane splits based on local canonical correlation coefficients calculated during training. Unlike axis-aligned alternatives, the decision surfaces of CCFs are not restricted to the coordinate system of the inputs features and therefore more naturally represent data with correlated inputs. CCFs naturally accommodate multiple outputs, provide a similar computational complexity to random forests, and inherit their impressive robustness to the choice of input parameters. As part of the CCF training algorithm, we also introduce projection bootstrapping, a novel alternative to bagging for oblique decision tree ensembles which maintains use of the full dataset in selecting split points, often leading to improvements in predictive accuracy. Our experiments show that, 

In [53]:
%%time

top_hyphenated_words(corpus_per_decade, topn=10)

CPU times: user 5.75 s, sys: 12.9 ms, total: 5.76 s
Wall time: 5.77 s


[('state-of-the-art', 13618),
 ('large-scale', 7345),
 ('real-time', 5187),
 ('real-world', 4757),
 ('x-ray', 4721),
 ('long-term', 4089),
 ('three-dimensional', 4001),
 ('two-dimensional', 3944),
 ('inline-formula', 3797),
 ('high-resolution', 3113)]

In [54]:
print_counts(corpus_per_decade, [
    "vector-space", "word-vector", "vector space", "word vector", "embedding space", "embedding-space",
    "word embedding", "word-embedding", "operator algebra", "operator-algebra"])

vector-space: 26
word-vector: 1
vector space: 302
word vector: 35
embedding space: 255
embedding-space: 0
word embedding: 292
word-embedding: 18
operator algebra: 41
operator-algebra: 0


In [55]:
corpus_per_decade = replace_hyphen_terms(
    corpus_per_decade, 
    ["vector-space", "vector-space.", "vector-space,", "vector-space)"], repl = " ")

In [56]:
nlp = init_spacy()

In [58]:
%%time

nlp_per_decade = dict()
for k, v in tqdm(corpus_per_decade.items()):
    nlp_per_decade[k] = spacy_docs(v, nlp)

  0%|          | 0/11 [00:00<?, ?it/s]

CPU times: user 36min 14s, sys: 8.1 s, total: 36min 22s
Wall time: 36min 27s


In [61]:
corpus_per_decade_lemmatized = dict()

for k,v in tqdm(nlp_per_decade.items()):
    corpus_per_decade_lemmatized[k] = get_lemmas_and_non_stop_from_nlp(v)

  0%|          | 0/11 [00:00<?, ?it/s]

In [67]:
len(corpus_per_decade_lemmatized)

11

In [68]:
corpus_lemmatized = [
    doc
    for txts_decade in corpus_per_decade_lemmatized.values()
    for doc in txts_decade
]

print(len(corpus_lemmatized))
corpus_lemmatized[0]

336160


['electrodynamic', 'general', 'relativity', 'theory']

In [64]:
corpus_decade_index = create_per_decade_index(corpus_per_decade_lemmatized)
corpus_decade_index

{1920: (0, 10),
 1930: (10, 42),
 1940: (42, 105),
 1950: (105, 515),
 1960: (515, 2049),
 1970: (2049, 5511),
 1980: (5511, 12112),
 1990: (12112, 32268),
 2000: (32268, 114873),
 2010: (114873, 276124),
 2020: (276124, 336160)}

In [65]:
len(corpus_per_decade_lemmatized[2020])

60036

In [69]:
%%time

corpus_bi = ngram_builder(corpus_lemmatized)
corpus_tri = ngram_builder(corpus_bi)
corpus_tetra = ngram_builder(corpus_tri)

CPU times: user 1min 30s, sys: 934 ms, total: 1min 31s
Wall time: 1min 32s


In [70]:
len(corpus_tetra)

336160

In [71]:
corpus_bi[-1][:10]

['edge',
 'label',
 'inference',
 'generalized',
 'stochastic_block',
 'model',
 ':',
 'spectral',
 'theory',
 'impossibility_result']

In [72]:
corpus_tri[-1][:10]

['edge',
 'label',
 'inference',
 'generalized',
 'stochastic_block',
 'model',
 ':',
 'spectral',
 'theory',
 'impossibility_result']

In [73]:
corpus_tetra[-1][:10]

['edge',
 'label',
 'inference',
 'generalized',
 'stochastic_block',
 'model',
 ':',
 'spectral_theory',
 'impossibility_result',
 '.']

In [74]:
global_model_corpus = corpus_tetra

In [75]:
# If you already have corpus_tri in memory, uncomment:
write_corpus(global_model_corpus, "../data/openalex_phrased.txt")

# ---------- (B) Stream sentences from disk (memory-safe) ----------
sentences = LineSentence("../data/openalex_phrased.txt")  # one tokenized doc per line

In [80]:
workers = max(1, mp.cpu_count() - 1)
epochs = 10

In [ ]:
%%time

# ---------- (C) Train word2vec ----------
w2v = fit_model(
    sentences, 
    "word2vec", 
    vector_size=300,
    window=5,
    min_count=10,
    sg=1,
    negative=10,
    sample=1e-4,
    epochs=epochs,
    workers=workers,
    callbacks=[EpochLogger(epochs)]
)

Epoch 1/10 | elapsed: 0.4 min | ETA: 3.8 min
Epoch 2/10 | elapsed: 0.8 min | ETA: 3.1 min
Epoch 3/10 | elapsed: 1.1 min | ETA: 2.7 min
Epoch 4/10 | elapsed: 1.5 min | ETA: 2.3 min
Epoch 5/10 | elapsed: 1.9 min | ETA: 1.9 min


In [ ]:
%%time

out_dir = "../models/w2v_openalex_baseline")
save_model(w2v, out_dir, model_name="word2vec")

In [ ]:
queries = ["vector_space", "vector-space", "hilbert_space", "banach_space", 
          "embedding", "vector", "embed_space", "token"]

In [ ]:
sanity_check(w2v, queries)

In [ ]:
%%time

# ---------- (C) Train FastText ----------
ft = fit_model(
    sentences, 
    "fasttext", 
    vector_size=300,
    window=5,
    min_count=10,
    sg=1,
    negative=10,
    sample=1e-4,
    epochs=epochs,
    workers=workers,
    callbacks=[EpochLogger(epochs)]
)

## Diachronic

In [85]:
corpus_per_decade_phrased = split_corpus_by_decade(global_model_corpus, corpus_decade_index)

In [86]:
corpus_per_decade_phrased[2020][0]

['canonical_correlation_forest',
 '.',
 'introduce',
 'canonical_correlation_forest',
 '(',
 'CCFs',
 ')',
 ',',
 'new',
 'decision_tree',
 'ensemble',
 'method',
 'classification_regression',
 '.',
 'individual',
 'canonical_correlation',
 'tree',
 'binary_decision_tree',
 'hyperplane',
 'split',
 'base',
 'local',
 'canonical_correlation',
 'coefficient',
 'calculate',
 'training',
 '.',
 'unlike',
 'axis-aligned',
 'alternative',
 ',',
 'decision',
 'surface',
 'ccf',
 'restrict',
 'coordinate_system',
 'input_feature',
 'naturally_represent',
 'datum',
 'correlate',
 'input',
 '.',
 'ccf',
 'naturally_accommodate',
 'multiple_output',
 ',',
 'provide',
 'similar',
 'computational_complexity',
 'random_forest',
 ',',
 'inherit',
 'impressive',
 'robustness',
 'choice',
 'input_parameter',
 '.',
 'CCF',
 'training',
 'algorithm',
 ',',
 'introduce',
 'projection',
 'bootstrapping',
 ',',
 'novel',
 'alternative',
 'bag',
 'oblique_decision_tree',
 'ensemble',
 'maintain',
 'use',
 'd